# Modeling

Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import joblib
from tqdm import tqdm
from sklearn.model_selection import TimeSeriesSplit

Load WTI Data

In [6]:
wti = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v3.parquet")

Load Tweet Data

In [7]:
tweets = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6.parquet")

In [8]:
tweets.shape

(15242, 53)

In [9]:
tweets.columns

Index(['timestamp_utc', 'clean_text', 'embedding', 'finbert_positive',
       'finbert_neutral', 'finbert_negative', 'finbert_sentiment_score',
       'finbert_entropy', 'finbert_confidence', 'financial_uncertainty_score',
       'financial_risk_sentiment', 'finbert_polarization', 'caps_ratio',
       'exclamation_count', 'exclamation_count_log', 'tweet_length',
       'tweet_length_log', 'market_aggression_index',
       'time_since_last_tweet_min', 'rolling_tweet_frequency_60m',
       'rolling_tweet_frequency_6h', 'rolling_tweet_frequency_24h',
       'tweet_burst_indicator', 'tweet_acceleration_6h',
       'sentiment_delta_vs_previous', 'rolling_sentiment_mean_60m',
       'rolling_sentiment_std_60m', 'is_trump_president', 'wti_bullish_score',
       'wti_bearish_score', 'energy_supply_score',
       'geopolitical_oil_risk_score', 'usd_strength_pressure_score',
       'fed_monetary_policy_score', 'risk_sentiment_score',
       'trump_energy_policy_score', 'trump_market_shock_langua

In [10]:
wti.shape

(2601773, 76)

In [11]:
wti.columns

Index(['timestamp_utc', 'open', 'high', 'low', 'close', 'log_close', 'ret_1',
       'hl_range', 'oc_return', 'ret_5', 'ret_15', 'ret_30', 'ret_60',
       'ret_240', 'mom_5', 'mom_15', 'mom_60', 'vol_close_5',
       'vol_parkinson_5', 'vol_close_15', 'vol_parkinson_15', 'vol_close_30',
       'vol_parkinson_30', 'vol_close_60', 'vol_parkinson_60', 'vol_close_240',
       'vol_parkinson_240', 'vol_gk', 'vol_ratio_5_60', 'sma_5', 'ema_5',
       'dist_ema_5', 'sma_15', 'ema_15', 'dist_ema_15', 'sma_60', 'ema_60',
       'dist_ema_60', 'sma_240', 'ema_240', 'dist_ema_240', 'hl_momentum',
       'body', 'upper_wick', 'lower_wick', 'abs_ret', 'vol_cluster', 'vol_5',
       'jump', 'hour', 'dayofweek', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'us_session', 'vol_rank', 'vol_regime', 'trend_60',
       'trend_regime', 'y_up_2', 'y_down_2', 'y_up_5', 'y_down_5', 'y_up_15',
       'y_down_15', 'y_up_30', 'y_down_30', 'y_up_60', 'y_down_60', 'y_up_240',
       'y_down_240', 'y_up_72

## Train-Validation-Test-Split

In [12]:
tweets_pre2020 = tweets[tweets['timestamp_utc'] < '2020-01-01'] # wird für training und validation verwendet
tweets_post2020 = tweets[tweets['timestamp_utc'] >= '2020-01-01'] # als test set für die out of sample performance

## ML-Pipeline

### 1. Config

In [13]:
PCA_COMPONENTS = 20
N_SPLITS = 5
PURGE_MINUTES = 1440

TIME_WTI = "timestamp_utc"
TIME_TWEET = "timestamp_utc"

### 2. Embedding Parser

In [14]:
def parse_embedding(x):
    return np.fromstring(x.strip("[]"), sep=" ")

### 3. Tweet Preprocessing + Embeddings

In [15]:
def prepare_tweets(df):

    df = df.copy()

    df[TIME_TWEET] = pd.to_datetime(df[TIME_TWEET], utc=True)

    df["embedding_vec"] = df["embedding"].apply(parse_embedding)

    emb = np.vstack(df["embedding_vec"].values)

    emb_cols = [f"emb_{i}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)

    df = pd.concat([df.reset_index(drop=True), emb_df], axis=1)

    return df

### 4. Tweet Aggregation

Notiz: Hier fehlen noch die aggregation für die restlichen spalten! also unvollständig

In [45]:
def aggregate_tweets(df):

    df = df.copy()

    df["minute"] = df[TIME_TWEET].dt.floor("min")

    emb_cols = [c for c in df.columns if c.startswith("emb_")]

    agg_dict = {

        # =========================
        # FinBERT
        # =========================
        "finbert_positive": "mean",
        "finbert_neutral": "mean",
        "finbert_negative": "mean",
        "finbert_sentiment_score": "mean",
        "finbert_entropy": "mean",
        "finbert_confidence": "max",
        "finbert_polarization": "max",

        # =========================
        # Financial NLP Scores
        # =========================
        "financial_uncertainty_score": "max",
        "financial_risk_sentiment": "mean",

        # =========================
        # Text Style
        # =========================
        "caps_ratio": "mean",
        "exclamation_count": "max",
        "exclamation_count_log": "max",

        "tweet_length": "mean",
        "tweet_length_log": "mean",

        "market_aggression_index": "max",

        # =========================
        # Tweet Dynamics
        # =========================
        "time_since_last_tweet_min": "min",

        "rolling_tweet_frequency_60m": "max",
        "rolling_tweet_frequency_6h": "max",
        "rolling_tweet_frequency_24h": "max",

        "tweet_burst_indicator": "max",
        "tweet_acceleration_6h": "max",

        # =========================
        # Sentiment Dynamics
        # =========================
        "sentiment_delta_vs_previous": "mean",
        "rolling_sentiment_mean_60m": "mean",
        "rolling_sentiment_std_60m": "mean",

        # =========================
        # Presidency Regime
        # =========================
        "is_trump_president": "max",

        # =========================
        # Oil / Macro Scores
        # =========================
        "wti_bullish_score": "max",
        "wti_bearish_score": "max",

        "energy_supply_score": "max",
        "geopolitical_oil_risk_score": "max",
        "usd_strength_pressure_score": "max",
        "fed_monetary_policy_score": "max",
        "risk_sentiment_score": "max",

        "trump_energy_policy_score": "max",
        "trump_market_shock_language_score": "max",
        "policy_shock_score": "max",

        # =========================
        # Topic Metrics
        # =========================
        "topic_concentration": "mean",

        "topic_energy_supply": "max",
        "topic_fed_monetary_policy": "max",
        "topic_geopolitical_oil_risk": "max",
        "topic_policy_shock": "max",
        "topic_risk_sentiment": "max",
        "topic_trump_energy_policy": "max",
        "topic_trump_market_shock_language": "max",
        "topic_usd_strength_pressure": "max",
        "topic_wti_bearish": "max",
        "topic_wti_bullish": "max",

        # =========================
        # Semantic Change
        # =========================
        "semantic_global_novelty": "max",
        "semantic_local_change": "max",
        "semantic_rolling_novelty": "max",
    }

    # =========================
    # Embeddings
    # =========================
    for c in emb_cols:
        agg_dict[c] = "mean"

    agg = (
        df.groupby("minute")
          .agg(agg_dict)
          .reset_index()
    )

    return agg

In [ ]:
# def aggregate_tweets(df):

#     df = df.copy()

#     df["minute"] = df[TIME_TWEET].dt.floor("min")

#     emb_cols = [c for c in df.columns if c.startswith("emb_")]

#     agg_dict = {
#         "finbert_sentiment_score": ["mean", "std"],
#         "financial_uncertainty_score": ["mean", "max"],
#         "tweet_length": ["mean", "sum"],
#         "exclamation_count": ["mean"]
#     }

#     for c in emb_cols:
#         agg_dict[c] = ["mean", "max", "min"]

#     agg = df.groupby("minute").agg(agg_dict)

#     agg.columns = ["_".join(col) for col in agg.columns]
#     agg = agg.reset_index()

#     return agg

### 5. Leakage-Free WTI Mapping

In [17]:
def map_to_wti(tweet_agg, wti):

    tweet_agg = tweet_agg.copy()
    wti = wti.copy()

    tweet_agg[TIME_TWEET] = pd.to_datetime(tweet_agg["minute"], utc=True)
    wti[TIME_WTI] = pd.to_datetime(wti[TIME_WTI], utc=True)

    # KEY RULE:
    # tweet -> previous completed minute candle
    tweet_agg["wti_time"] = tweet_agg["minute"] - pd.Timedelta(minutes=1)

    df = tweet_agg.merge(
        wti,
        left_on="wti_time",
        right_on=TIME_WTI,
        how="left"
    )

    df = df.dropna(subset=["open"])

    return df

### 6. Feature / Target Split

In [18]:
TARGETS = [c for c in wti.columns if c.startswith("y_")]

def split_xy(df):

    y = df[TARGETS]
    X = df.drop(columns=TARGETS)

    return X, y

### 7. Purged CV

## nutze lieber time series split - ist eine getestete bibliothek

In [ ]:
# class PurgedCV:

#     def __init__(self, n_splits=5, purge=1440):
#         self.n_splits = n_splits
#         self.purge = purge

#     def split(self, X):

#         n = len(X)
#         idx = np.arange(n)
#         fold = n // self.n_splits

#         for i in range(self.n_splits):

#             val_start = i * fold
#             val_end = n if i == self.n_splits - 1 else (i + 1) * fold

#             val_idx = idx[val_start:val_end]

#             train_mask = np.ones(n, dtype=bool)

#             train_mask[
#                 max(0, val_start - self.purge):
#                 min(n, val_end + self.purge)
#             ] = False

#             train_idx = idx[train_mask]

#             yield train_idx, val_idx

### 8. Models

In [19]:
MODELS = {
    "logreg": LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=5000,
        class_weight="balanced"
    ),

    "rf": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42
    ),

    "mlp": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=300
    )
}

### 9. Evaluation

In [20]:
def evaluate(y_true, pred, prob):

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "f1": f1_score(y_true, pred),
        "acc": accuracy_score(y_true, pred)
    }

In [21]:
def clean_features(df):

    # print(df.isna().sum()[df.isna().sum() > 0])

    # nur numerische Features
    df = df.select_dtypes(include=[np.number])

    # df = df.dropna()

    return df


## Start: Durchführung einiger Tests und Korrekturen

In [46]:
tweets_pre2020.shape

(9831, 53)

In [52]:
tweets_pre2020

,timestamp_utc,clean_text,embedding,finbert_positive,finbert_neutral,finbert_negative,finbert_sentiment_score,finbert_entropy,finbert_confidence,financial_uncertainty_score,...,topic_policy_shock,topic_risk_sentiment,topic_trump_energy_policy,topic_trump_market_shock_language,topic_usd_strength_pressure,topic_wti_bearish,topic_wti_bullish,semantic_global_novelty,semantic_local_change,semantic_rolling_novelty
0,2017-01-20 00:40:51+00:00,Thank you for joining us at the Lincoln Memori...,[ 1.44826015e-02 2.63881516e-02 5.50416000e-...,0.726080,0.262663,0.011257,0.714823,0.634073,0.726080,0.273920,...,0,0,1,0,0,0,0,0.000000,0.000000,0.000000
1,2017-01-20 04:24:33+00:00,Thank you for a wonderful evening in Washingto...,[-2.39205379e-02 8.10888410e-02 1.08877219e-...,0.067150,0.910896,0.021954,0.045196,0.350208,0.910896,0.089104,...,0,0,0,1,0,0,0,0.370935,0.370935,0.370935
2,2017-01-20 12:31:53+00:00,It all begins today! I will see you at 11:00 A...,[-4.61930446e-02 -1.01754069e-02 4.10163514e-...,0.069542,0.904756,0.025702,0.043839,0.370044,0.904756,0.095244,...,1,0,0,0,0,0,0,0.531908,0.627273,0.531908
3,2017-01-20 17:51:25+00:00,Today we are not merely transferring power fro...,[-1.74501184e-02 -5.02789803e-02 4.17382782e-...,0.063037,0.919632,0.017331,0.045706,0.321566,0.919632,0.080368,...,1,0,0,0,0,0,0,0.785248,0.863338,0.785248
4,2017-01-20 17:51:58+00:00,"power from Washington, D.C. and giving it back...",[-4.91272621e-02 8.73695835e-02 1.05445653e-...,0.051730,0.926997,0.021273,0.030457,0.305389,0.926997,0.073003,...,1,0,0,0,0,0,0,0.365552,0.559879,0.365552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9826,2019-12-31 22:11:14+00:00,"The Fake News said I played golf today, and I ...",[ 9.72597674e-02 2.62575373e-02 3.68273235e-...,0.036121,0.509062,0.454817,-0.418697,0.821996,0.509062,0.490938,...,0,0,0,1,0,0,0,0.664997,0.805404,0.672579
9827,2019-12-31 22:24:36+00:00,I am far better on HealthCare than the Democra...,[ 1.88880824e-02 6.23342395e-02 -9.03689244e-...,0.281979,0.687251,0.030770,0.251210,0.721838,0.687251,0.312749,...,0,0,1,0,0,0,0,0.629043,0.846071,0.673144
9828,2019-12-31 23:19:43+00:00,"Get this straightened out, Governor @GavinNews...",[-3.38741094e-02 5.17788567e-02 1.01585187e-...,0.037804,0.916663,0.045533,-0.007728,0.344251,0.916663,0.083337,...,1,0,0,0,0,0,0,0.656069,0.808496,0.627686
9829,2019-12-31 23:19:49+00:00,Wonderful account of U.S. Embassy (Iraq) vs. t...,[-1.15986366e-03 6.48995265e-02 -8.25404841e-...,0.062249,0.900076,0.037675,0.024573,0.391126,0.900076,0.099924,...,0,0,0,1,0,0,0,0.610798,0.709188,0.425103


In [55]:
print(tweets_agg.isna().sum().to_string())

minute                               0
finbert_positive                     0
finbert_neutral                      0
finbert_negative                     0
finbert_sentiment_score              0
finbert_entropy                      0
finbert_confidence                   0
finbert_polarization                 0
financial_uncertainty_score          0
financial_risk_sentiment             0
caps_ratio                           0
exclamation_count                    0
exclamation_count_log                0
tweet_length                         0
tweet_length_log                     0
market_aggression_index              0
time_since_last_tweet_min            0
rolling_tweet_frequency_60m          0
rolling_tweet_frequency_6h           0
rolling_tweet_frequency_24h          0
tweet_burst_indicator                0
tweet_acceleration_6h                0
sentiment_delta_vs_previous          0
rolling_sentiment_mean_60m           0
rolling_sentiment_std_60m            0
is_trump_president       

In [49]:
tweets_agg = aggregate_tweets(tweets_df)


C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_12616\2612417994.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


In [67]:
tweets_df

,timestamp_utc,clean_text,embedding,finbert_positive,finbert_neutral,finbert_negative,finbert_sentiment_score,finbert_entropy,finbert_confidence,financial_uncertainty_score,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,2017-01-20 00:40:51+00:00,Thank you for joining us at the Lincoln Memori...,[ 1.44826015e-02 2.63881516e-02 5.50416000e-...,0.726080,0.262663,0.011257,0.714823,0.634073,0.726080,0.273920,...,0.053353,0.035265,0.008177,0.008031,0.063478,0.026637,0.115716,0.047976,-0.096644,0.008382
1,2017-01-20 04:24:33+00:00,Thank you for a wonderful evening in Washingto...,[-2.39205379e-02 8.10888410e-02 1.08877219e-...,0.067150,0.910896,0.021954,0.045196,0.350208,0.910896,0.089104,...,0.030188,0.024219,0.009116,-0.034081,0.049663,0.108277,0.067122,0.077678,-0.019405,0.012493
2,2017-01-20 12:31:53+00:00,It all begins today! I will see you at 11:00 A...,[-4.61930446e-02 -1.01754069e-02 4.10163514e-...,0.069542,0.904756,0.025702,0.043839,0.370044,0.904756,0.095244,...,0.065073,0.060351,0.031749,-0.007576,-0.044693,0.009487,0.111785,-0.035808,-0.056803,-0.012898
3,2017-01-20 17:51:25+00:00,Today we are not merely transferring power fro...,[-1.74501184e-02 -5.02789803e-02 4.17382782e-...,0.063037,0.919632,0.017331,0.045706,0.321566,0.919632,0.080368,...,0.059997,-0.014752,-0.019062,0.047931,-0.060031,-0.012779,0.046233,0.010037,0.026718,-0.048252
4,2017-01-20 17:51:58+00:00,"power from Washington, D.C. and giving it back...",[-4.91272621e-02 8.73695835e-02 1.05445653e-...,0.051730,0.926997,0.021273,0.030457,0.305389,0.926997,0.073003,...,0.111034,0.063763,-0.006465,0.020676,-0.018112,0.017635,0.016711,0.019296,-0.017491,-0.005452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9826,2019-12-31 22:11:14+00:00,"The Fake News said I played golf today, and I ...",[ 9.72597674e-02 2.62575373e-02 3.68273235e-...,0.036121,0.509062,0.454817,-0.418697,0.821996,0.509062,0.490938,...,0.041582,0.055684,0.020491,-0.060781,-0.010578,-0.044808,0.012371,-0.094054,-0.095364,0.012135
9827,2019-12-31 22:24:36+00:00,I am far better on HealthCare than the Democra...,[ 1.88880824e-02 6.23342395e-02 -9.03689244e-...,0.281979,0.687251,0.030770,0.251210,0.721838,0.687251,0.312749,...,0.002517,0.020357,0.018597,-0.076485,0.030023,-0.007431,0.091620,-0.007423,0.052559,0.008719
9828,2019-12-31 23:19:43+00:00,"Get this straightened out, Governor @GavinNews...",[-3.38741094e-02 5.17788567e-02 1.01585187e-...,0.037804,0.916663,0.045533,-0.007728,0.344251,0.916663,0.083337,...,0.045007,-0.027046,0.004636,-0.019306,-0.060638,0.103975,0.011237,-0.010050,-0.022501,0.039709
9829,2019-12-31 23:19:49+00:00,Wonderful account of U.S. Embassy (Iraq) vs. t...,[-1.15986366e-03 6.48995265e-02 -8.25404841e-...,0.062249,0.900076,0.037675,0.024573,0.391126,0.900076,0.099924,...,0.159909,0.049244,-0.028145,-0.036363,0.040118,0.112681,0.079833,0.060450,-0.002610,0.003170


In [56]:
tweets_agg

,minute,finbert_positive,finbert_neutral,finbert_negative,finbert_sentiment_score,finbert_entropy,finbert_confidence,finbert_polarization,financial_uncertainty_score,financial_risk_sentiment,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,2017-01-20 00:40:00+00:00,0.726080,0.262663,0.011257,0.714823,0.634073,0.726080,0.714823,0.273920,0.011257,...,0.053353,0.035265,0.008177,0.008031,0.063478,0.026637,0.115716,0.047976,-0.096644,0.008382
1,2017-01-20 04:24:00+00:00,0.067150,0.910896,0.021954,0.045196,0.350208,0.910896,0.045196,0.089104,0.021954,...,0.030188,0.024219,0.009116,-0.034081,0.049663,0.108277,0.067122,0.077678,-0.019405,0.012493
2,2017-01-20 12:31:00+00:00,0.069542,0.904756,0.025702,0.043839,0.370044,0.904756,0.043839,0.095244,0.025702,...,0.065073,0.060351,0.031749,-0.007576,-0.044693,0.009487,0.111785,-0.035808,-0.056803,-0.012898
3,2017-01-20 17:51:00+00:00,0.057384,0.923314,0.019302,0.038082,0.313478,0.926997,0.045706,0.080368,0.019302,...,0.085516,0.024505,-0.012764,0.034304,-0.039072,0.002428,0.031472,0.014666,0.004614,-0.026852
4,2017-01-20 17:52:00+00:00,0.029348,0.915398,0.055254,-0.025906,0.344478,0.915398,0.025906,0.084602,0.055254,...,0.002116,-0.009099,-0.030534,0.037991,-0.007711,-0.039436,-0.021810,0.031594,0.084712,-0.066602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9086,2019-12-31 21:38:00+00:00,0.057532,0.815734,0.126734,-0.069202,0.592206,0.815734,0.069202,0.184266,0.126734,...,0.106933,0.072919,0.024955,-0.025920,-0.031492,0.028797,0.029399,-0.014722,0.010664,0.013035
9087,2019-12-31 22:11:00+00:00,0.036121,0.509062,0.454817,-0.418697,0.821996,0.509062,0.418697,0.490938,0.454817,...,0.041582,0.055684,0.020491,-0.060781,-0.010578,-0.044808,0.012371,-0.094054,-0.095364,0.012135
9088,2019-12-31 22:24:00+00:00,0.281979,0.687251,0.030770,0.251210,0.721838,0.687251,0.251210,0.312749,0.030770,...,0.002517,0.020357,0.018597,-0.076485,0.030023,-0.007431,0.091620,-0.007423,0.052559,0.008719
9089,2019-12-31 23:19:00+00:00,0.050027,0.908369,0.041604,0.008423,0.367688,0.916663,0.024573,0.099924,0.041604,...,0.102458,0.011099,-0.011755,-0.027835,-0.010260,0.108328,0.045535,0.025200,-0.012556,0.021439


In [58]:
tweets_agg["minute"].is_monotonic_increasing

True

In [59]:
df = map_to_wti(tweets_agg, wti)


In [64]:
tweets_agg.shape

(9091, 434)

In [70]:
tweets_agg.dtypes

minute                     datetime64[us, UTC]
finbert_positive                       float64
finbert_neutral                        float64
finbert_negative                       float64
finbert_sentiment_score                float64
                                  ...         
emb_379                                float64
emb_380                                float64
emb_381                                float64
emb_382                                float64
emb_383                                float64
Length: 434, dtype: object

In [69]:
wti.dtypes

timestamp_utc    datetime64[us, UTC]
open                         float64
high                         float64
low                          float64
close                        float64
                        ...         
y_down_240                     int64
y_up_720                       int64
y_down_720                     int64
y_up_1440                      int64
y_down_1440                    int64
Length: 76, dtype: object

In [ ]:
print(wti.isna().sum().to_string())

timestamp_utc        0
open                 0
high                 0
low                  0
close                0
log_close            0
ret_1                0
hl_range             0
oc_return            0
ret_5                0
ret_15               0
ret_30               0
ret_60               0
ret_240              0
mom_5                0
mom_15               0
mom_60               0
vol_close_5          0
vol_parkinson_5      0
vol_close_15         0
vol_parkinson_15     0
vol_close_30         0
vol_parkinson_30     0
vol_close_60         0
vol_parkinson_60     0
vol_close_240        0
vol_parkinson_240    0
vol_gk               0
vol_ratio_5_60       0
sma_5                0
ema_5                0
dist_ema_5           0
sma_15               0
ema_15               0
dist_ema_15          0
sma_60               0
ema_60               0
dist_ema_60          0
sma_240              0
ema_240              0
dist_ema_240         0
hl_momentum          0
body                 0
upper_wick 

In [76]:
wti.shape

(2601773, 76)

In [73]:
full_range = pd.date_range(wti[TIME_WTI].min(), wti[TIME_WTI].max(), freq="min")
missing = len(full_range) - wti[TIME_WTI].nunique()
missing

1602965

In [72]:
wti[TIME_WTI].dt.second.value_counts().head()
wti[TIME_WTI].dt.microsecond.value_counts().head()

timestamp_utc
0    2601773
Name: count, dtype: int64

In [71]:
wti

,timestamp_utc,open,high,low,close,log_close,ret_1,hl_range,oc_return,ret_5,...,y_up_30,y_down_30,y_up_60,y_down_60,y_up_240,y_down_240,y_up_720,y_down_720,y_up_1440,y_down_1440
0,2015-01-01 23:00:00+00:00,53.750,53.880,53.750,53.850,3.986202,0.002417,0.002414,0.001859,0.002603,...,0,1,1,0,1,0,0,1,0,1
1,2015-01-01 23:01:00+00:00,53.830,53.840,53.750,53.750,3.984344,-0.001859,0.001674,-0.001487,0.000558,...,0,1,1,0,1,0,0,1,0,1
2,2015-01-01 23:02:00+00:00,53.740,53.740,53.660,53.670,3.982854,-0.001489,0.001491,-0.001303,-0.000931,...,1,0,1,0,1,0,0,1,0,1
3,2015-01-01 23:03:00+00:00,53.680,53.710,53.650,53.680,3.983040,0.000186,0.001118,0.000000,-0.000931,...,1,0,1,0,1,0,0,1,0,1
4,2015-01-01 23:04:00+00:00,53.700,53.700,53.640,53.640,3.982295,-0.000745,0.001119,-0.001118,-0.001490,...,1,0,1,0,1,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2601768,2022-12-30 21:53:00+00:00,80.407,80.407,80.367,80.392,4.386915,-0.000249,0.000498,-0.000187,0.000871,...,0,0,0,0,0,0,0,0,0,0
2601769,2022-12-30 21:54:00+00:00,80.402,80.427,80.387,80.422,4.387288,0.000373,0.000497,0.000249,-0.000311,...,0,0,0,0,0,0,0,0,0,0
2601770,2022-12-30 21:55:00+00:00,80.417,80.447,80.402,80.407,4.387101,-0.000187,0.000560,-0.000124,-0.000497,...,0,0,0,0,0,0,0,0,0,0
2601771,2022-12-30 21:56:00+00:00,80.427,80.427,80.369,80.387,4.386852,-0.000249,0.000722,-0.000497,-0.000933,...,0,0,0,0,0,0,0,0,0,0


In [68]:
df

,minute,finbert_positive,finbert_neutral,finbert_negative,finbert_sentiment_score,finbert_entropy,finbert_confidence,finbert_polarization,financial_uncertainty_score,financial_risk_sentiment,...,y_up_30,y_down_30,y_up_60,y_down_60,y_up_240,y_down_240,y_up_720,y_down_720,y_up_1440,y_down_1440
1,2017-01-20 04:24:00+00:00,0.067150,0.910896,0.021954,0.045196,0.350208,0.910896,0.045196,0.089104,0.021954,...,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0
2,2017-01-20 12:31:00+00:00,0.069542,0.904756,0.025702,0.043839,0.370044,0.904756,0.043839,0.095244,0.025702,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,2017-01-20 17:51:00+00:00,0.057384,0.923314,0.019302,0.038082,0.313478,0.926997,0.045706,0.080368,0.019302,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
4,2017-01-20 17:52:00+00:00,0.029348,0.915398,0.055254,-0.025906,0.344478,0.915398,0.025906,0.084602,0.055254,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
5,2017-01-20 17:53:00+00:00,0.126149,0.860709,0.013142,0.113007,0.447201,0.860709,0.113007,0.139291,0.013142,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9082,2019-12-31 19:41:00+00:00,0.080208,0.653630,0.266161,-0.185953,0.631308,0.859849,0.485754,0.480829,0.266161,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
9083,2019-12-31 20:15:00+00:00,0.037788,0.827257,0.134955,-0.097167,0.550956,0.827257,0.097167,0.172743,0.134955,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
9084,2019-12-31 21:19:00+00:00,0.311054,0.543744,0.145202,0.165853,0.832973,0.636526,0.494696,0.478133,0.145202,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
9085,2019-12-31 21:34:00+00:00,0.078465,0.891184,0.030351,0.048114,0.408443,0.891184,0.048114,0.108816,0.030351,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


## Ende

### 10. Core Training Loop

To Dos:
- aggregate tweets macht das noch nicht mit allen features, muss ich nachbessern
- bei feature_sets werden aktuell nur die embeddings gedroppt und baseline enthält damit noch tweet features
- wird nicht pca transfomiert stand jetzt
- feature selection
- feature importance + diagnostik fehlt
- hyperparameter tuning fehlt
- modelle müssen angepasst werden - MLP bewusster konfigurieren etc., early stopping etc. pp
- overfitting detecten und beheben
- andere metriken nehmen
- in dataframe wegschreiben und im nachgang analysierenm, auch feature importance, residen, testing etc.
- Visualisierungen bei timerseries split, learning curves etc.
- Residuenanalye + Diagnostik!
- ggf das finale model nochmal auf kompletten Daten trainieren
- bei der tweet aggegration haben die std cokumns sher viele nullwerte, sollte mN gff komplett drauf verzichten
- problem: es fehlen viele minutendaten und einfach rolling werte zu nehmen gibt zeitliche abhängigkietn falsch wieder, mappingmlogik sollte daher sein: wenn letzte wti minute nicht verfgbar dann mappe auf letzte verfügbare wti kurs der länger als eine minute zurück liegt


### Alternativer Ansatz:

In [ ]:
def run_pipeline(wti_df, tweets_df):

    results = []
    models_store = {}

    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)
    
    # Beginn von mir ergänzer Code
    std_cols = [c for c in X.columns if c.endswith("_std")]
    # print(std_cols)
    for c in std_cols:
        X[c] = X[c].fillna(0)
    # Ende von mir ergänzer Code

    # feature_sets = {
    #     "baseline": X.drop(columns=[c for c in X.columns if c.startswith("emb_")]),
    #     "enhanced": X
    # }
    feature_sets = {
        "baseline": clean_features(
            X.drop(columns=[c for c in X.columns if c.startswith("emb_")])
        ),
        "enhanced": clean_features(X)
    }
    # cv = PurgedCV(n_splits=N_SPLITS, purge=PURGE_MINUTES)
    cv = TimeSeriesSplit(
        n_splits=N_SPLITS,
        gap=PURGE_MINUTES
    )

    for target in TARGETS:

        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():

            for model_name, model in MODELS.items():

                for fold, (tr, va) in enumerate(cv.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    # print(X_train.columns)
                    # print(X_train.dtypes)
                    # X_train.drop("minute", axis=1, inplace=True)
                    # X_val.drop("minute", axis = 1, inplace = True)

                    scaler = StandardScaler()
                    X_train = scaler.fit_transform(X_train)
                    X_val = scaler.transform(X_val)

                    model.fit(X_train, y_train)

                    prob = model.predict_proba(X_val)[:, 1]
                    pred = (prob > 0.5).astype(int)

                    scores = evaluate(y_val, pred, prob)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        **scores
                    })

                    models_store[(target, fs_name, model_name, fold)] = model

    return pd.DataFrame(results), models_store

### Bisher verwendeter Training Loop

In [ ]:
def run_pipeline(wti_df, tweets_df):

    results = []
    models_store = {}

    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)
    
    # Beginn von mir ergänzer Code
    std_cols = [c for c in X.columns if c.endswith("_std")]
    # print(std_cols)
    for c in std_cols:
        X[c] = X[c].fillna(0)
    # Ende von mir ergänzer Code

    # feature_sets = {
    #     "baseline": X.drop(columns=[c for c in X.columns if c.startswith("emb_")]),
    #     "enhanced": X
    # }
    feature_sets = {
        "baseline": clean_features(
            X.drop(columns=[c for c in X.columns if c.startswith("emb_")])
        ),
        "enhanced": clean_features(X)
    }
    # cv = PurgedCV(n_splits=N_SPLITS, purge=PURGE_MINUTES)
    cv = TimeSeriesSplit(
        n_splits=N_SPLITS,
        gap=PURGE_MINUTES
    )

    for target in TARGETS:

        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():

            for model_name, model in MODELS.items():

                for fold, (tr, va) in enumerate(cv.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    # print(X_train.columns)
                    # print(X_train.dtypes)
                    # X_train.drop("minute", axis=1, inplace=True)
                    # X_val.drop("minute", axis = 1, inplace = True)

                    scaler = StandardScaler()
                    X_train = scaler.fit_transform(X_train)
                    X_val = scaler.transform(X_val)

                    model.fit(X_train, y_train)

                    prob = model.predict_proba(X_val)[:, 1]
                    pred = (prob > 0.5).astype(int)

                    scores = evaluate(y_val, pred, prob)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        **scores
                    })

                    models_store[(target, fs_name, model_name, fold)] = model

    return pd.DataFrame(results), models_store

hier fehlt noch die training scores auszuwerten, nicht nur die validations damit man overfitting etc. erkennen kann

### 11. Best Model Export

In [56]:
def save_best(results, models_store):

    best = results.sort_values("roc_auc", ascending=False).iloc[0]

    key = (
        best.target,
        best.feature_set,
        best.model,
        best.fold
    )

    joblib.dump(models_store[key], "best_model.pkl")

    return best

In [74]:
std_cols = [c for c in tweets_pre2020.columns if c.endswith("_std")]
print(std_cols)

[]


### 12. RUN

In [77]:
results, models = run_pipeline(wti, tweets_pre2020)

best_model_info = save_best(results, models)

print(best_model_info)

C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_17404\1248267005.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg = agg.reset_index()


Series([], dtype: int64)
Series([], dtype: int64)


c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use 

target         y_down_2
feature_set    baseline
model                rf
fold                  4
roc_auc        0.600463
f1             0.095745
acc            0.702016
Name: 39, dtype: object


In [84]:
results.to_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\results_v1.parquet", index=False)

In [89]:
len(models)

480